In [5]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def main():
    print("=== STARTING MODEL TRAINING PIPELINE WITH 80/20 SPLIT & RFE ===")
    os.makedirs("models", exist_ok=True)
    os.makedirs("data", exist_ok=True)
    
    data_path = "data/sales_data.csv"
    if not os.path.exists(data_path) and os.path.exists("sales_data.csv"):
        import shutil
        shutil.copy("sales_data.csv", data_path)
        
    df = pd.read_csv(data_path)
    df["Order_Date"] = pd.to_datetime(df["Order_Date"])
    df["Order_Month"] = df["Order_Date"].dt.month
    df["Order_DayOfWeek"] = df["Order_Date"].dt.dayofweek

    num_cols = ["Quantity", "Unit_Price_INR", "Discount", "Order_Month", "Order_DayOfWeek"]
    cat_cols = ["Category", "Subcategory", "Region", "Segment", "Channel", "Payment_Method"]

    X = df[num_cols + cat_cols]
    y_reg = df["Profit_INR"]
    y_cls = (df["Returned"] == "Yes").astype(int)

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
        ]
    )

    # --- REGRESSION TASK (80/20 Split & RFE) ---
    print("\n--- Training Regression Model (80/20 Split + RFE) ---")
    X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
        X, y_reg, test_size=0.2, random_state=42
    )

    reg_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("rfe", RFE(estimator=LinearRegression(), n_features_to_select=45)),
        ("model", LinearRegression())
    ])
    
    reg_pipeline.fit(X_train_reg, y_train_reg)
    reg_preds = reg_pipeline.predict(X_test_reg)
    
    r2_final = r2_score(y_test_reg, reg_preds)
    mae_final = mean_absolute_error(y_test_reg, reg_preds)
    rmse_final = np.sqrt(mean_squared_error(y_test_reg, reg_preds))

    print(f"Regression Results -> R²: {r2_final:.4f} | MAE: ₹{mae_final:.2f} | RMSE: ₹{rmse_final:.2f}")
    joblib.dump(reg_pipeline, "models/profit_regression.joblib")

    # --- CLASSIFICATION TASK (80/20 Stratified Split & RFE) ---
    print("\n--- Training Classification Model (80/20 Stratified Split + RFE) ---")
    X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
        X, y_cls, test_size=0.2, stratify=y_cls, random_state=42
    )

    clf_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("rfe", RFE(estimator=LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42), n_features_to_select=30)),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
    ])

    clf_pipeline.fit(X_train_cls, y_train_cls)
    clf_preds = clf_pipeline.predict(X_test_cls)

    acc = accuracy_score(y_test_cls, clf_preds)
    prec = precision_score(y_test_cls, clf_preds, zero_division=0)
    rec = recall_score(y_test_cls, clf_preds, zero_division=0)
    f1 = f1_score(y_test_cls, clf_preds, zero_division=0)

    print(f"Classification Results -> Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    joblib.dump(clf_pipeline, "models/return_classifier.joblib")
    print("\nModels successfully trained, evaluated, and saved to models/")

if __name__ == "__main__":
    main()

=== STARTING MODEL TRAINING PIPELINE WITH 80/20 SPLIT & RFE ===

--- Training Regression Model (80/20 Split + RFE) ---
Regression Results -> R²: 0.8233 | MAE: ₹467.79 | RMSE: ₹746.47

--- Training Classification Model (80/20 Stratified Split + RFE) ---
Classification Results -> Accuracy: 0.5357 | Precision: 0.0777 | Recall: 0.4932 | F1: 0.1342

Models successfully trained, evaluated, and saved to models/
